In [1]:
import numpy as np
import pandas as pd
import json
from ssb import StrategicSegmentBuilder, StrategicSegmentScore

def generate_strategic_engine_stress_data(n_samples: int = 50000, high_zero_inflation: bool = True) -> pd.DataFrame:
    """Generates a highly non-linear, structurally challenging synthetic dataset 
    designed to test all parsing and pipeline edge cases in ssb.py.
    """
    np.random.seed(42)
    
    # 1. Base Target Component (Binary Event)
    # If high_zero_inflation is requested, we use a sparse target rate (~1.5%)
    target_rate = 0.015 if high_zero_inflation else 0.10
    target = np.random.choice([0, 1], size=n_samples, p=[1 - target_rate, target_rate])
    
    # 2. Stress Feature: Scientific Notation & Extreme Values
    # Forces optbinning to output string intervals like '[1.0e-05, 5.0e+06)'
    # Tests: StrategicSegmentBuilder._is_numeric_string()
    num_scientific = np.random.uniform(1e-6, 1e7, size=n_samples)
    # Inject extreme outliers to force wide float boundaries
    num_scientific[::100] = 9.9e8 
    num_scientific[1::100] = 1.1e-7

    # 3. Stress Feature: Heavy Structural Missingness
    # Forces optbinning to isolate a 'Missing' / 'Special' binning state
    # Tests: StrategicSegmentBuilder.parse_rule_to_sql() -> "IS NULL" branch
    num_with_nulls = np.random.normal(loc=100, scale=25, size=n_samples)
    num_with_nulls_df = pd.Series(num_with_nulls)
    # Inject 35% explicitly missing values (NaN maps to NULL in DuckDB via fetchnumpy)
    num_with_nulls_df.iloc[np.random.choice(n_samples, size=int(n_samples * 0.35), replace=False)] = np.nan

    # 4. Stress Feature: Categorical Cardinality with Common Grouping
    # Forces optbinning to generate multi-item categorical arrays like "['Tier_A', 'Tier_B']"
    # Tests: StrategicSegmentBuilder.parse_rule_to_sql() -> ast.literal_eval() path
    cat_pool = ['Tier_A', 'Tier_B', 'Tier_C', 'Tier_D', 'Unmapped_Merchant', 'Legacy_State']
    # Build a strong multi-item relationship to target to force bin grouping
    cat_raw = []
    for t in target:
        if t == 1:
            cat_raw.append(np.random.choice(['Tier_A', 'Tier_B'], p=[0.7, 0.3]))
        else:
            cat_raw.append(np.random.choice(cat_pool, p=[0.1, 0.1, 0.3, 0.3, 0.1, 0.1]))
    
    # 5. Diversity Conflict Features
    # Features built to be highly predictive but belonging to the same functional group
    # Tests: StrategicSegmentBuilder.is_diverse() and enable_diversity routing
    risk_score_alpha = np.random.uniform(300, 850, size=n_samples)
    risk_score_beta = risk_score_alpha * np.random.uniform(0.9, 1.1, size=n_samples) # Strongly correlated
    
    # Force high lift pockets in risk scores to guarantee they get selected if not pruned
    risk_score_alpha[target == 1] = np.random.uniform(300, 450, size=np.sum(target == 1))
    risk_score_beta[target == 1] = np.random.uniform(300, 450, size=np.sum(target == 1))

    # Assemble final DataFrame
    df = pd.DataFrame({
        "account_id": [f"ACC_{str(i).zfill(8)}" for i in range(n_samples)],
        "target_event": target,
        "num_scientific": num_scientific,
        "num_with_nulls": num_with_nulls_df,
        "merchant_tier": cat_raw,
        "risk_score_alpha": risk_score_alpha,
        "risk_score_beta": risk_score_beta,
        "noise_feature": np.random.randn(n_samples) # Completely uninformative feature
    })
    
    return df

In [2]:
if __name__ == "__main__":
    import os
    
    print("Step 1: Generating Rigorous Edge-Case Dataset...")
    test_df = generate_strategic_engine_stress_data(n_samples=50000, high_zero_inflation=True)
    
    print(f"Dataset Shape: {test_df.shape}")
    print(f"Total Target Events: {test_df['target_event'].sum()} (Rate: {test_df['target_event'].mean():.2%})")
    print(f"Missing Values in 'num_with_nulls': {test_df['num_with_nulls'].isna().sum()}\n")

    # Define strict business categories to test diversity constraints
    feature_groups = {
        "internal_risk_tier": ["risk_score_alpha", "risk_score_beta"],
        "external_behavior": ["num_scientific", "merchant_tier"]
    }

    print("Step 2: Testing StrategicSegmentBuilder Framework...")
    builder = StrategicSegmentBuilder(
        target="target_event",
        n_jobs=-1,
        min_sample_size=500,       # Lowered slightly to guarantee rule extraction over sparse target
        min_lift=2.5,
        min_events=10,
        top_n_vars=10,
        max_segments=5,
        max_feature_reuse=1,       # Stricts feature dominance tracking
        enable_diversity=True,     # Forces pruning of risk_score_beta if alpha is adopted
        feature_groups=feature_groups,
        ignore_features=["account_id", "noise_feature"]
    )

    # Run Segment Extraction
    extracted_segments = builder.extract_segments(test_df)
    print(f"\nSuccessfully Extracted {len(extracted_segments)} Segments.")
    print(json.dumps(extracted_segments, indent=2))

    # Evaluate Coverage
    coverage_report = builder.evaluate_final_coverage(test_df)
    print("\n--- Segment Coverage Performance Matrix ---")
    for row in coverage_report:
        print(f"Segment {row['segment']}: Count={row['total_count']:,} | Rate={row['response_rate']:.2f}% | Lift={row['lift']:.2f}x")

    print("\nStep 3: Creating Binary Segment Columns for Scorecard Engine...")
    # Map the rule filters back into columns to pass into the scorecard scorer
    segment_cols = []
    for seg in extracted_segments:
        col_name = f"seg_flag_{seg['segment_id']}"
        # Execute query natively inside a temporary DuckDB block to derive indicators
        import duckdb
        ctx = duckdb.connect()
        ctx.execute("CREATE TABLE tmp AS SELECT * FROM test_df")
        res_ids = ctx.execute(f"SELECT account_id FROM tmp WHERE {seg['sql_filter']}").fetchall()
        matching_ids = {r[0] for r in res_ids}
        
        test_df[col_name] = test_df["account_id"].apply(lambda x: 1 if x in matching_ids else 0)
        segment_cols.append(col_name)

    print(f"Configured Segment Dummy Matrix Columns: {segment_cols}")

    print("\nStep 4: Testing High Zero-Inflation Scorecard & Deciling...")
    scorer = StrategicSegmentScore(
        target_col="target_event",
        primary_key="account_id",
        segment_cols=segment_cols
    )

    # Runs tracking weights, matrix dot products, and reverse sorted deciles
    model_artifact = scorer.calculate_and_export_weights(test_df, export_path="stress_test_scorecard.json")
    
    print("\n--- Final Model Metadata ---")
    print(json.dumps(model_artifact["model_metadata"], indent=2))
    print("\n--- Decile Minimum Cutoffs ---")
    print(json.dumps(model_artifact["decile_min_thresholds"], indent=2))
    
    # Cleanup file output
    if os.path.exists("stress_test_scorecard.json"):
        os.remove("stress_test_scorecard.json")
    print("\n[SUCCESS] Engine successfully completed stress-testing against all target edge cases!")

Step 1: Generating Rigorous Edge-Case Dataset...


2026-07-08 16:01:58,208 | INFO     | [ssb.py:364] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-08 16:01:58,253 | INFO     | [ssb.py:133] | Feature group validation passed. (4 features mapped)
2026-07-08 16:01:58,255 | INFO     | [ssb.py:402] | Iteration 1 | Remaining Volume: 50,000 | Base Rate: 1.55%


Dataset Shape: (50000, 8)
Total Target Events: 774 (Rate: 1.55%)
Missing Values in 'num_with_nulls': 17500

Step 2: Testing StrategicSegmentBuilder Framework...


2026-07-08 16:01:58,968 | INFO     | [ssb.py:502] | Feature Usage Tracker Update -> 'merchant_tier' used count = 1
2026-07-08 16:01:58,969 | INFO     | [ssb.py:502] | Feature Usage Tracker Update -> 'risk_score_beta' used count = 1
2026-07-08 16:01:58,969 | INFO     | [ssb.py:517] | Segment 1 Captured (Size Floor: 500 | Lift Floor: 2.5): merchant_tier IN ('Tier_A') AND (risk_score_beta >= 332.43 AND risk_score_beta < 386.40)
2026-07-08 16:01:58,973 | INFO     | [ssb.py:402] | Iteration 2 | Remaining Volume: 49,351 | Base Rate: 1.17%
2026-07-08 16:01:59,538 | INFO     | [ssb.py:502] | Feature Usage Tracker Update -> 'risk_score_alpha' used count = 1
2026-07-08 16:01:59,538 | INFO     | [ssb.py:517] | Segment 2 Captured (Size Floor: 500 | Lift Floor: 2.5): (risk_score_alpha >= 365.02 AND risk_score_alpha < 392.78)
2026-07-08 16:01:59,541 | INFO     | [ssb.py:402] | Iteration 3 | Remaining Volume: 46,868 | Base Rate: 0.97%
2026-07-08 16:02:00,084 | INFO     | [ssb.py:488] | No active cand


Successfully Extracted 2 Segments.
[
  {
    "segment_id": 1,
    "rule_string": "merchant_tier=<ArrowStringArray>\n['Tier_A']\nLength: 1, dtype: str & risk_score_beta=[332.43, 386.40)",
    "sql_filter": "merchant_tier IN ('Tier_A') AND (risk_score_beta >= 332.43 AND risk_score_beta < 386.40)",
    "count": 648,
    "rate": 30.09259259259259,
    "lift": 19.439659297540434,
    "meta_applied_sample_size": 500,
    "meta_applied_min_lift": 2.5
  },
  {
    "segment_id": 2,
    "rule_string": "risk_score_alpha=[365.02, 392.78)",
    "sql_filter": "(risk_score_alpha >= 365.02 AND risk_score_alpha < 392.78)",
    "count": 2481,
    "rate": 4.9173720274083035,
    "lift": 4.198567939872443,
    "meta_applied_sample_size": 500,
    "meta_applied_min_lift": 2.5
  }
]

--- Segment Coverage Performance Matrix ---
Segment 0: Count=46,868 | Rate=0.97% | Lift=0.63x
Segment 1: Count=649 | Rate=30.20% | Lift=19.51x
Segment 2: Count=2,483 | Rate=4.91% | Lift=3.17x

Step 3: Creating Binary Segment C

2026-07-08 16:02:00,350 | INFO     | [ssb.py:624] | Computing harmonic scorecard weights...
2026-07-08 16:02:00,350 | INFO     | [ssb.py:661] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-08 16:02:00,354 | INFO     | [ssb.py:678] | Dataset Zero-Inflation Rate: 98.45%
2026-07-08 16:02:00,355 | INFO     | [ssb.py:681] | High Zero-Inflation (>=80%). Isolating Active Population...
2026-07-08 16:02:00,356 | INFO     | [ssb.py:693] | Calibrating deciles across 3,132 target customers...



--- Final Model Metadata ---
{
  "total_training_population": 50000,
  "active_scored_population": 3132,
  "active_population_pct": 6.26,
  "baseline_event_rate": 0.0155
}

--- Decile Minimum Cutoffs ---
{
  "1": 537,
  "2": 537,
  "3": 37,
  "4": 37,
  "5": 37,
  "6": 37,
  "7": 37,
  "8": 37,
  "9": 37,
  "10": 37
}

[SUCCESS] Engine successfully completed stress-testing against all target edge cases!
